# Tessera | Portable Test Center

**Kullanım:** Colab'da aç → `Runtime → Run all`. Kod hücreleri gizli. Testler otomatik çalışır; hatalar aşağıdaki **Sorun Havuzu**'na eklenir.

### Kapsam
- GCC ve Clang strict mock backend testleri
- ASan / UBSan
- Test edilen commit SHA, ortam, loglar ve hata raporu

> Bu testler gerçek Windows hook, unwind, DLL lifecycle, MSVC veya Detours doğrulaması değildir.

### Erişim
Colab Secrets'ta `GH_TOKEN` tanımla. Fine-grained token yalnız `Jue-dev/Tessera` için **Contents: Read-only** olsun. Notebook access açık olmalı. Token'ı hücrelere yapıştırma.


In [ ]:
#@title 1 — Repository bağlantısı { display-mode: "form" }
from google.colab import userdata
import os, subprocess, pathlib, shutil, time, json, zipfile, platform, datetime, html
from IPython.display import display, HTML
import ipywidgets as widgets

REPO = "https://github.com/Jue-dev/Tessera.git"
BRANCH = "issue-6/unwind-lifecycle"
WORKDIR = pathlib.Path("/content/tessera")
LOG_DIR = pathlib.Path("/content/tessera-test-report")
LOG_DIR.mkdir(exist_ok=True)
ISSUE_POOL = []
records = []
HEAD_SHA = "NOT_CHECKED_OUT"

try:
    token = userdata.get("GH_TOKEN")
except Exception as exc:
    token = None
    ISSUE_POOL.append({"stage":"GitHub secret","status":"FAIL","detail":f"Secret erişimi: {type(exc).__name__}: {exc}"})

if not token:
    ISSUE_POOL.append({"stage":"GitHub authentication","status":"FAIL","detail":"GH_TOKEN bulunamadı. Colab Secrets'ta ekle ve notebook access'i etkinleştir."})
else:
    if WORKDIR.exists():
        shutil.rmtree(WORKDIR)
    askpass = pathlib.Path("/content/tessera_askpass.sh")
    # Create a real multi-line script. Keep it alive for the entire git clone.
    askpass.write_text('''#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' 'x-access-token' ;;
  *Password*) printf '%s\\n' "$GIT_PASSWORD" ;;
  *) printf '%s\\n' "$GIT_PASSWORD" ;;
esac
''', encoding="utf-8")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update({"GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT":"0", "GIT_PASSWORD":token})
    try:
        clone = subprocess.run(["git","clone","--depth","1","--single-branch","--branch",BRANCH,REPO,str(WORKDIR)],env=env,capture_output=True,text=True,timeout=180)
    except Exception as exc:
        clone = None
        ISSUE_POOL.append({"stage":"Repository clone","status":"FAIL","detail":f"{type(exc).__name__}: {exc}"})
    finally:
        askpass.unlink(missing_ok=True)
        env.pop("GIT_PASSWORD",None)
        token = None
    if clone is not None and clone.returncode != 0:
        # Avoid echoing credential-like strings. Git stderr should not contain the token,
        # but only retain the last portion and redact it defensively.
        err = clone.stderr[-3000:]
        if 'token' in locals() and token:
            err = err.replace(token,"[REDACTED]")
        ISSUE_POOL.append({"stage":"Repository clone","status":"FAIL","detail":err or f"git clone exited {clone.returncode}"})
    elif clone is not None and clone.returncode == 0:
        HEAD_SHA = subprocess.check_output(["git","rev-parse","HEAD"],cwd=WORKDIR,text=True).strip()
        print(f"Repository hazır ✓ | {BRANCH} | {HEAD_SHA[:12]}")

if HEAD_SHA == "NOT_CHECKED_OUT":
    print("Repository bağlantısı başarısız. Hata Sorun Havuzu'na eklendi; notebook rapor hücresine kadar devam edecek.")
else:
    print("Bağlantı başarılı. Testler sonraki hücrede çalışacak.")

## Testler

Aşağıdaki otomatik testler çalıştırılır. Bir test başarısız olsa bile diğer testler devam eder; böylece tek raporda tüm sorunları görebiliriz.

In [ ]:
#@title 2 — Otomatik testler { display-mode: "form" }
TESTS = [
    ("GCC · strict mock", ["bash","verify/windows_mock/run_windows_mock_tests.sh"], {}),
    ("Clang · strict mock", ["bash","verify/windows_mock/run_windows_mock_tests.sh"], {"CXX":"clang++"}),
    ("GCC · ASan + UBSan", ["bash","verify/windows_mock/run_windows_mock_tests.sh"], {
        "EXTRA_CXXFLAGS":"-fsanitize=address,undefined -fno-omit-frame-pointer",
        "ASAN_OPTIONS":"detect_leaks=1:halt_on_error=1",
        "UBSAN_OPTIONS":"halt_on_error=1:print_stacktrace=1"
    }),
]
if HEAD_SHA == "NOT_CHECKED_OUT":
    for name, command, overrides in TESTS:
        records.append({"test":name,"status":"SKIPPED","exit_code":None,"duration_seconds":0,"log":"repository unavailable"})
    print("⏭️ Repository yok; testler atlandı. Clone hatası rapora dahil edilecek.")
else:
    for name, command, overrides in TESTS:
        started=time.time()
        test_env=os.environ.copy(); test_env.update(overrides)
        log_file=LOG_DIR/(name.lower().replace(" · ","_").replace(" ","_") + ".log")
        try:
            proc=subprocess.run(command,cwd=WORKDIR,env=test_env,capture_output=True,text=True,timeout=600)
            status="PASS" if proc.returncode==0 and "WINDOWS_MOCK_VALIDATION_COMPLETE" in proc.stdout else "FAIL"
            log=f"COMMAND: {' '.join(command)}\nEXIT: {proc.returncode}\n\n--- STDOUT ---\n{proc.stdout}\n--- STDERR ---\n{proc.stderr}"
            rec={"test":name,"status":status,"exit_code":proc.returncode,"duration_seconds":round(time.time()-started,2),"log":str(log_file)}
        except subprocess.TimeoutExpired as exc:
            status="FAIL"; log=f"TIMEOUT after 600 seconds\nSTDOUT:\n{exc.stdout}\nSTDERR:\n{exc.stderr}"
            rec={"test":name,"status":status,"exit_code":124,"duration_seconds":600,"log":str(log_file)}
        except Exception as exc:
            status="FAIL"; log=f"EXCEPTION: {type(exc).__name__}: {exc}"
            rec={"test":name,"status":status,"exit_code":-1,"duration_seconds":round(time.time()-started,2),"log":str(log_file)}
        log_file.write_text(log,encoding="utf-8")
        records.append(rec)
        if status != "PASS":
            ISSUE_POOL.append({"stage":name,"status":"FAIL","detail":log[-12000:]})
        print(("✅ " if status=="PASS" else "❌ ")+name+f" — {rec['duration_seconds']} sn")
print("Test aşaması tamamlandı.")

## Sonuç ve paylaşılabilir hata raporu

Bu hücre; commit bilgisi, ortam, test sonuçları ve hata loglarını bir araya getirir. Rapor dosyası oluşturulur ve indirme bağlantısı sunulur.

In [ ]:
#@title 3 — Sonuç ve kopyalanabilir Sorun Havuzu { display-mode: "form" }
summary={"project":"Tessera","branch":BRANCH,"commit":HEAD_SHA,"commit_url":(f"https://github.com/Jue-dev/Tessera/commit/{HEAD_SHA}" if HEAD_SHA!="NOT_CHECKED_OUT" else None),"timestamp_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),"platform":platform.platform(),"python":platform.python_version(),"tests":records,"issues":ISSUE_POOL,"overall":"PASS" if HEAD_SHA!="NOT_CHECKED_OUT" and records and all(x["status"]=="PASS" for x in records) and not ISSUE_POOL else "FAIL"}
(LOG_DIR/"summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")

lines=["TESSERA PORTABLE TEST REPORT",f"Overall: {summary['overall']}",f"Branch: {BRANCH}",f"Commit: {HEAD_SHA}",f"Commit URL: {summary['commit_url'] or 'not available'}",f"Timestamp UTC: {summary['timestamp_utc']}",f"Environment: {summary['platform']}","", "TEST RESULTS:"]
for x in records: lines.append(f"[{x['status']}] {x['test']} | exit={x['exit_code']} | {x['duration_seconds']}s")
lines += ["", "ISSUE POOL:"]
for i,x in enumerate(ISSUE_POOL,1): lines += [f"--- ISSUE {i}: {x['stage']} [{x['status']}] ---",x['detail'],"--- END ISSUE ---"]
report_text="\n".join(lines)
(LOG_DIR/"CHAT_REPORT.txt").write_text(report_text,encoding="utf-8")
for x in records:
    p=pathlib.Path(x.get("log",""))
    if p.exists():
        with open(p,encoding="utf-8",errors="replace") as f: lines += ["",f"===== LOG: {x['test']} =====",f.read()[-12000:]]
full_report="\n".join(lines)
(LOG_DIR/"FULL_DIAGNOSTIC_REPORT.txt").write_text(full_report,encoding="utf-8")
archive=pathlib.Path("/content/Tessera_Test_Report.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for f in LOG_DIR.glob("*"): z.write(f,arcname=f.name)

status_color="#16803c" if summary['overall']=="PASS" else "#b42318"
display(HTML(f"<div style='padding:14px;border:1px solid #ccc;border-radius:12px'><h2 style='color:{status_color}'>Tessera Test: {summary['overall']}</h2><p><b>Commit:</b> {HEAD_SHA}</p><p><b>Hata sayısı:</b> {len(ISSUE_POOL)}</p><p>Rapor aşağıdaki kutuda. Kutunun içine tıkla → Ctrl+A → Ctrl+C; ardından buraya yapıştır.</p></div>"))
copy_box=widgets.Textarea(value=report_text,description="Rapor",layout=widgets.Layout(width="100%",height="360px"))
display(copy_box)
print("Rapor ayrıca runtime dosyalarında: /content/Tessera_Test_Report.zip")
try:
    from google.colab import files
    files.download(str(archive))
except Exception as exc:
    print("ZIP indirme otomatik açılamadı; kopyalama kutusunu kullan.",exc)
if summary['overall']!="PASS": print("❗ Başarısızlıklar ISSUE_POOL içinde toplandı. Yukarıdaki raporu kopyalayıp sohbete gönder.")

### Bana aktarım

Son hücredeki rapor kutusuna tıkla, `Ctrl+A` → `Ctrl+C` ve buraya yapıştır. Rapor; klonlama, test ve ortam hatalarını bir arada içerir. İstersen ZIP de oluşturulur, ama artık indirme zorunlu değil.

Notebook token'ı rapora eklemez.

In [ ]:
#@title (Gizli) Notebook yardımcı kodları
# Colab form hücresi; kullanıcıdan işlem gerektirmez.